In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

urls = [
    "https://www.uia.no/english/studies/courses/2026/spring/ikt469.html"
]

courses = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

def clean_text(tag):
    if not tag:
        return None
    return " ".join(tag.get_text(" ", strip=True).split())

# sections you do NOT want
skip_sections = {
    "contact",
    "resources",
    "about_this_site",
}

for url in urls:
    print(f"Fetching: {url}")

    response = requests.get(url, headers=headers, timeout=15)
    print("Status:", response.status_code)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    course_data = {"url": url}

    # Title
    h1 = soup.find("h1")
    course_data["title"] = clean_text(h1)

    # Fact fields
    for dt in soup.find_all("dt"):
        key = clean_text(dt)
        dd = dt.find_next_sibling("dd")
        value = clean_text(dd)

        if key and value:
            normalized_key = (
                key.lower()
                .replace(":", "")
                .replace(" ", "_")
                .replace("-", "_")
            )
            course_data[normalized_key] = value

    # H2 sections
    for h2 in soup.find_all("h2"):
        section_name = clean_text(h2)
        if not section_name:
            continue

        normalized_section = (
            section_name.lower()
            .replace(":", "")
            .replace(" ", "_")
            .replace("-", "_")
        )

        # skip unwanted sections
        if normalized_section in skip_sections:
            continue

        section_parts = []
        for sib in h2.find_next_siblings():
            if sib.name == "h2":
                break

            if sib.name in ["p", "ul", "ol", "div"]:
                text = sib.get_text(" ", strip=True)
                if text:
                    section_parts.append(text)

        if section_parts:
            course_data[normalized_section] = "\n".join(section_parts)

    courses.append(course_data)
    print("Done with page")

df = pd.DataFrame(courses)
df.to_csv("uia_ikt_courses.csv", index=False, encoding="utf-8-sig")

print(df.T)

Fetching: https://www.uia.no/english/studies/courses/2026/spring/ikt469.html
Status: 200
Done with page
                                                                                                    0
url                                                 https://www.uia.no/english/studies/courses/202...
title                                                       IKT469 Deep Neural Networks (Spring 2026)
ects_credits                                                                                      7.5
responsible_department                                             Faculty of Engineering and Science
course_leader                                                                          Morten Goodwin
lecture_semester                                                                               Spring
teaching_language                                                                             English
duration                                                                        

In [ ]:
#4. Chunking and indexing RUN FROM HERE
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1) Load your scraped data
df = pd.read_csv("uia_ikt_courses.csv")

# 2) Turn each row into a LangChain Document
documents = []

for i, row in df.iterrows():
    title = str(row.get("title", ""))
    ects = str(row.get("ects", ""))
    leader = str(row.get("course_leader", ""))

    # If you also scraped more fields later, add them here
    content = f"""
Title: {row['title']}
ECTS: {row['ects']}
Course leader: {row['course_leader']}

Learning outcomes:
{row.get('learning_outcomes', '')}

Course contents:
{row.get('contents', '')}
""".strip()

    documents.append(
        Document(
            page_content=content,
            metadata={
                "row_id": i,
                "title": title,
                "ects": ects,
                "course_leader": leader,
            },
        )
    )

# 3) Chunk the documents
# For course descriptions, small chunks are usually enough
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Chunks created: {len(chunks)}")

# 4) Create a CPU-only embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 5) Index chunks in local Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
)

print("Chunking and indexing complete.")